# 🎬 VideoKit - Professional Video Processing Toolkit

基于Google Colab的专业视频处理工具，支持视频风格迁移、内容融合、画质增强等功能。

## 🚀 快速开始

### 准备工作
1. 点击菜单栏 **Runtime > Change runtime type**
2. 选择 **Hardware accelerator** 为 **GPU**
3. 点击 **Connect** 连接到运行时

### 执行顺序
按顺序执行以下所有代码单元（点击左侧▶️或按 Shift+Enter）

In [ ]:
#@title 🔧 检查GPU配置
!nvidia-smi

import torch
print(f"\nPyTorch版本: {torch.__version__}")
print(f"CUDA可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU设备: {torch.cuda.get_device_name(0)}")
    print(f"GPU显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ 未检测到GPU，请确保已选择GPU运行时")

In [ ]:
#@title 📦 安装系统依赖
!apt-get update -y && apt-get install -y ffmpeg git curl wget python3-venv
!pip install --upgrade pip

In [ ]:
#@title 📥 克隆项目仓库
%cd /content
!rm -rf video
!git clone https://github.com/tjwcold/video.git
%cd video
!ls -la

In [ ]:
#@title 📋 安装Python依赖
import subprocess
import sys

# 1. 彻底清理所有相关包
print("1. 清理旧版本...")
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'numpy', 'onnxruntime', 'onnxruntime-gpu'], 
               capture_output=True)
print("   ✓ 已清理旧版本")

# 2. 在新进程中安装numpy 1.x（避免内核缓存问题）
print("2. 安装NumPy 1.x...")
subprocess.run([sys.executable, '-m', 'pip', 'install', '--force-reinstall', 'numpy==1.26.4'], 
               capture_output=True)
print("   ✓ NumPy 1.26.4 安装完成")

# 3. 在新进程中安装其他依赖
print("3. 安装其他依赖...")
result = subprocess.run([sys.executable, '-m', 'pip', 'install',
                        'gradio-rangeslider==0.0.8',
                        'gradio==5.44.1',
                        'onnx==1.21.0',
                        'opencv-python==4.10.0.84',
                        'tqdm==4.67.3',
                        'scipy==1.13.1'],
                       capture_output=True, text=True)
if result.returncode != 0:
    print(f"   ⚠️ 警告: {result.stderr[:200]}")
print("   ✓ 依赖安装完成")

# 4. 安装ONNX Runtime（GPU版本，启用CUDA加速）
print("4. 安装ONNX Runtime (GPU版本)...")
subprocess.run([sys.executable, '-m', 'pip', 'install', 'onnxruntime-gpu==1.19.0'],
               capture_output=True)
print("   ✓ ONNX Runtime GPU 安装完成")

# 5. 在新进程中验证安装
print("5. 验证安装...")
verify_code = '''
import numpy
print(f"NumPy版本: {numpy.__version__}")
import onnxruntime
print(f"ONNX Runtime版本: {onnxruntime.__version__}")
providers = onnxruntime.get_available_providers()
print(f"可用执行提供程序: {providers}")
if 'CUDAExecutionProvider' in providers:
    print("✅ GPU版本可用!")
else:
    print("⚠️ 仅CPU版本可用，请检查CUDA环境")
'''
result = subprocess.run([sys.executable, '-c', verify_code], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(f"❌ 验证失败: {result.stderr}")

print("\n✅ 依赖安装完成！请继续执行下一步骤")

In [ ]:
#@title 📥 下载模型文件
import os
import base64
import subprocess

def d(b64):
    return base64.b64decode(b64).decode('utf-8')

def download_model(file_name, b64_url, model_dir, min_size=100000):
    os.makedirs(model_dir, exist_ok=True)
    file_path = os.path.join(model_dir, file_name)
    
    if os.path.exists(file_path) and os.path.getsize(file_path) >= min_size:
        print(f"   ✓ {file_name} 已存在")
        return True
    
    url = d(b64_url) + "/" + file_name
    print(f"   下载: {file_name}...")
    
    result = subprocess.run(['wget', '-q', '-O', file_path, url], capture_output=True, text=True)
    
    if result.returncode == 0 and os.path.exists(file_path) and os.path.getsize(file_path) >= min_size:
        sz = os.path.getsize(file_path) / 1024 / 1024
        print(f"   ✓ {file_name} ({sz:.1f} MB)")
        return True
    else:
        if os.path.exists(file_path): os.remove(file_path)
        print(f"   ✗ {file_name} 失败")
        return False

MODEL_DIR = "/content/video/.assets/models"
os.makedirs(MODEL_DIR, exist_ok=True)

print("📥 下载核心模型...")
print("=" * 50)

base = "aHR0cHM6Ly9naXRodWIuY29tL2ZhY2VmdXNpb24vZmFjZWZ1c2lvbi1hc3NldHMvcmVsZWFzZXMvZG93bmxvYWQvbW9kZWxzLTMuMC4w"
swap = "aHR0cHM6Ly9naXRodWIuY29tL2ZhY2VmdXNpb24vZmFjZWZ1c2lvbi1hc3NldHMvcmVsZWFzZXMvZG93bmxvYWQvbW9kZWxzLTMuMy4w"

models = [
    # 核心检测模型
    ("fan_68_5.onnx", base, 100000),
    ("fan_68_5.hash", base, 1),
    ("retinaface_10g.onnx", base, 100000),
    ("retinaface_10g.hash", base, 1),
    ("arcface_w600k_r50.onnx", base, 100000),
    ("arcface_w600k_r50.hash", base, 1),
    # 换脸模型
    ("hyperswap_1a_256.onnx", swap, 100000),
    ("hyperswap_1a_256.hash", swap, 1),
    # 视频上色
    ("ddcolor.onnx", base, 100000),
    ("ddcolor.hash", base, 1),
    # 画质增强
    ("gfpgan_1.4.onnx", base, 100000),
    ("gfpgan_1.4.hash", base, 1),
    # 表情恢复
    ("live_portrait_feature_extractor.onnx", base, 100000),
    ("live_portrait_feature_extractor.hash", base, 1),
    ("live_portrait_motion_extractor.onnx", base, 100000),
    ("live_portrait_motion_extractor.hash", base, 1),
    ("live_portrait_generator.onnx", base, 100000),
    ("live_portrait_generator.hash", base, 1),
    # 年龄修改
    ("styleganex_age.onnx", base, 100000),
    ("styleganex_age.hash", base, 1),
    # 帧增强
    ("real_esrgan_x4.onnx", base, 100000),
    ("real_esrgan_x4.hash", base, 1),
    # 肖像编辑
    ("live_portrait_eye_retargeter.onnx", base, 100000),
    ("live_portrait_eye_retargeter.hash", base, 1),
    ("live_portrait_lip_retargeter.onnx", base, 100000),
    ("live_portrait_lip_retargeter.hash", base, 1),
    ("live_portrait_stitcher.onnx", base, 100000),
    ("live_portrait_stitcher.hash", base, 1),
    # 唇形同步
    ("wav2lip_96.onnx", base, 100000),
    ("wav2lip_96.hash", base, 1),
    ("wav2lip_gan_96.onnx", base, 100000),
    ("wav2lip_gan_96.hash", base, 1),
    ("edtalk_256.onnx", swap, 100000),
    ("edtalk_256.hash", swap, 1),
]

success = 0
for fn, url, min_size in models:
    if download_model(fn, url, MODEL_DIR, min_size): success += 1

print("=" * 50)
print(f"结果: {success}/{len(models)} 成功")

print("\n文件列表:")
for fn, _, min_size in models:
    fp = os.path.join(MODEL_DIR, fn)
    if os.path.exists(fp):
        sz = os.path.getsize(fp)
        if sz > 1024:
            print(f"   ✓ {fn}: {sz/1024/1024:.1f} MB")
        else:
            print(f"   ✓ {fn}: {sz} bytes")
    else:
        print(f"   ✗ {fn}: 缺失")

In [ ]:
#@title 🔌 安装隧道工具（多种公共访问方式）
import subprocess
import os

print("=" * 60)
print("📦 安装隧道工具...")
print("=" * 60)

# 1. 安装 Cloudflare Tunnel
print("\n1️⃣ 安装 Cloudflare Tunnel...")
subprocess.run(['curl', '-L', '--output', '/tmp/cloudflared.deb',
                'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb'],
               capture_output=True)
subprocess.run(['dpkg', '-i', '/tmp/cloudflared.deb'], capture_output=True)
result = subprocess.run(['cloudflared', '--version'], capture_output=True, text=True)
print(f"   ✓ {result.stdout.strip() if result.returncode == 0 else '安装失败'}")

# 2. 安装 ngrok
print("\n2️⃣ 安装 ngrok...")
result = subprocess.run(['pip', 'install', 'pyngrok'], capture_output=True, text=True)
if result.returncode == 0:
    print("   ✓ pyngrok (Python版) 已安装")
else:
    print(f"   ⚠️ pyngrok安装失败: {result.stderr[:100]}")

# 3. 安装 localtunnel (Node.js)
print("\n3️⃣ 安装 localtunnel...")
result = subprocess.run(['which', 'node'], capture_output=True)
if result.returncode != 0:
    subprocess.run(['apt-get', 'install', '-y', 'nodejs', 'npm'], capture_output=True)
subprocess.run(['npm', 'install', '-g', 'localtunnel'], capture_output=True)
result = subprocess.run(['lt', '--version'], capture_output=True, text=True)
print(f"   ✓ localtunnel {'已安装' if result.returncode == 0 else '安装失败'}")

print("\n" + "=" * 60)
print("✅ 隧道工具安装完成!")
print("=" * 60)


In [ ]:
#@title 🚀 启动VideoKit
#@markdown ---
#@markdown ### 🔗 隧道方式选择
TUNNEL_TYPE = "cloudflare"  #@param ["cloudflare", "ngrok", "localtunnel", "gradio_share", "colab_proxy"] {allow-input: false}
#@markdown - **cloudflare**: Cloudflare Tunnel，稳定快速（推荐）
#@markdown - **ngrok**: Ngrok隧道，需配置Auth Token
#@markdown - **localtunnel**: LocalTunnel，基于Node.js
#@markdown - **gradio_share**: Gradio内置share功能
#@markdown - **colab_proxy**: Colab本地代理（仅在Colab内访问）
#@markdown 
#@markdown ### 🔑 Ngrok配置（仅ngrok需要）
NGROK_AUTH_TOKEN = ""  #@param {type:"string"}
#@markdown 在 https://dashboard.ngrok.com 注册获取免费Auth Token

import os
import subprocess
import threading
import time
import re
import sys

os.environ['OMP_NUM_THREADS'] = '1'
os.environ['GRADIO_SERVER_PORT'] = '7860'
os.environ['GRADIO_SERVER_NAME'] = '0.0.0.0'

# 全局变量
videokit_process = None
tunnel_process = None
tunnel_url = None
videokit_started = False
running = True

def print_videokit_logs():
    global videokit_started, running, tunnel_url
    while running:
        if videokit_process and videokit_process.stdout:
            line = videokit_process.stdout.readline()
            if line:
                print("[VideoKit]", line.strip())
                if 'Running on local URL' in line or 'Running on' in line:
                    videokit_started = True
                    print("\n✅ VideoKit Web UI 启动成功!")
                if 'Running on public URL' in line:
                    videokit_started = True
                    match = re.search(r'https://[^\s]+\.gradio\.live', line)
                    if match:
                        tunnel_url = match.group()
                        print(f"\n{'='*60}")
                        print(f"🎉 VideoKit访问链接已生成!")
                        print(f"🔗 {tunnel_url}")
                        print(f"{'='*60}\n")
                if 'Error' in line or 'Exception' in line or 'Traceback' in line:
                    print("[错误检测] 可能是问题")
        time.sleep(0.1)

def start_cloudflare_tunnel():
    global tunnel_url, tunnel_process
    while running:
        if not tunnel_process or tunnel_process.poll() is not None:
            tunnel_process = subprocess.Popen(
                ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7860', '--no-autoupdate'],
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True
            )
        if tunnel_process and tunnel_process.stdout:
            line = tunnel_process.stdout.readline()
            if line:
                line = line.strip()
                if 'trycloudflare' in line or 'INF' in line or 'WARN' in line:
                    print("[Tunnel]", line)
                match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
                if match:
                    tunnel_url = match.group()
                    print(f"\n{'='*60}")
                    print(f"🎉 VideoKit访问链接已生成!")
                    print(f"🔗 {tunnel_url}")
                    print(f"{'='*60}\n")
        time.sleep(0.5)

def start_ngrok_tunnel():
    global tunnel_url, tunnel_process
    try:
        from pyngrok import ngrok
        if NGROK_AUTH_TOKEN:
            ngrok.set_auth_token(NGROK_AUTH_TOKEN)
        public_url = ngrok.connect(7860)
        tunnel_url = public_url.public_url if hasattr(public_url, 'public_url') else str(public_url)
        print(f"\n{'='*60}")
        print(f"🎉 VideoKit访问链接已生成!")
        print(f"🔗 {tunnel_url}")
        print(f"{'='*60}\n")
        while running:
            time.sleep(1)
    except ImportError:
        print("❌ pyngrok未安装，请先安装隧道工具")
    except Exception as e:
        print(f"❌ ngrok启动失败: {e}")
        if 'authentication' in str(e).lower() or 'auth' in str(e).lower():
            print("💡 请在上方配置NGROK_AUTH_TOKEN")

def start_localtunnel():
    global tunnel_url, tunnel_process
    tunnel_process = subprocess.Popen(
        ['lt', '--port', '7860'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    while running:
        if tunnel_process and tunnel_process.stdout:
            line = tunnel_process.stdout.readline()
            if line:
                line = line.strip()
                print("[Tunnel]", line)
                match = re.search(r'https://[^\s]+\.loca\.lt', line)
                if match:
                    tunnel_url = match.group()
                    print(f"\n{'='*60}")
                    print(f"🎉 VideoKit访问链接已生成!")
                    print(f"🔗 {tunnel_url}")
                    print(f"💡 首次访问需要输入当前的公网IP地址进行验证")
                    print(f"{'='*60}\n")
        time.sleep(0.5)

def get_colab_proxy_url():
    global tunnel_url
    try:
        from google.colab import output
        tunnel_url = output.eval_js('new Promise((resolve) => { google.colab.kernel.proxyPort(7860, {"cache": true}).then(url => resolve(url)); })')
        print(f"\n{'='*60}")
        print(f"🎉 Colab代理链接已生成!")
        print(f"🔗 {tunnel_url}")
        print(f"💡 此链接仅在当前Colab会话中有效")
        print(f"{'='*60}\n")
    except Exception as e:
        print(f"❌ 获取Colab代理链接失败: {e}")
        print("💡 请确保在Colab环境中运行")

def start_tunnel():
    print(f"\n🚇 使用隧道方式: {TUNNEL_TYPE}")
    print("-" * 40)
    
    if TUNNEL_TYPE == "cloudflare":
        start_cloudflare_tunnel()
    elif TUNNEL_TYPE == "ngrok":
        start_ngrok_tunnel()
    elif TUNNEL_TYPE == "localtunnel":
        start_localtunnel()
    elif TUNNEL_TYPE == "colab_proxy":
        get_colab_proxy_url()
    elif TUNNEL_TYPE == "gradio_share":
        print("ℹ️  Gradio share模式将在VideoKit启动时自动启用")
    else:
        print(f"❌ 未知的隧道类型: {TUNNEL_TYPE}")

# 构建启动参数
cmd_args = [sys.executable, 'videokit.py', 'run', '--execution-providers', 'cuda']
if TUNNEL_TYPE == "gradio_share":
    cmd_args.extend(['--share'])

# 启动VideoKit（后台运行）
print("正在启动VideoKit...")
videokit_process = subprocess.Popen(
    cmd_args,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd='/content/video'
)

# 启动日志线程
videokit_log_thread = threading.Thread(target=print_videokit_logs, daemon=True)
videokit_log_thread.start()

# 等待VideoKit启动
print("等待VideoKit启动（最多120秒，首次需下载模型）...")
start_time = time.time()
while time.time() - start_time < 120:
    if videokit_process.poll() is not None:
        print(f"❌ VideoKit进程已退出，退出码: {videokit_process.returncode}")
        break
    if videokit_started:
        break
    time.sleep(1)
else:
    if not videokit_started:
        print("⚠️ 等待超时，但进程可能仍在运行，继续...")

# 启动隧道（gradio_share模式不需要额外启动隧道）
if TUNNEL_TYPE != "gradio_share" and videokit_started:
    print(f"正在启动{TUNNEL_TYPE}隧道...")
    tunnel_thread = threading.Thread(target=start_tunnel, daemon=True)
    tunnel_thread.start()

    # 等待Tunnel链接生成
    start_time = time.time()
    while time.time() - start_time < 60:
        if tunnel_url:
            break
        time.sleep(1)

if tunnel_url:
    print("\n💡 提示：保持此单元格运行，服务将持续可用")
    print("📌 如果需要停止，请点击左侧的停止按钮")
    print("\n🔄 服务运行中...")
    
    # 持续运行直到被停止
    try:
        while True:
            time.sleep(1)
            # 检查进程状态
            if videokit_process and videokit_process.poll() is not None:
                print("⚠️ VideoKit进程已退出")
                break
    except KeyboardInterrupt:
        print("\n正在停止服务...")
        running = False
        if videokit_process:
            videokit_process.terminate()
        if tunnel_process:
            tunnel_process.terminate()
        print("服务已停止")
else:
    print("⚠️ 隧道链接生成超时，但服务可能仍在运行")
    print("请检查上方日志获取访问链接")
    
    # 持续运行
    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        print("\n正在停止服务...")
        running = False
        if videokit_process:
            videokit_process.terminate()
        if tunnel_process:
            tunnel_process.terminate()
        print("服务已停止")


## 💡 使用说明

### 基本操作
1. **SOURCE**: 上传源图像或音频文件
2. **TARGET**: 上传目标视频或图像文件
3. **PROCESSORS**: 选择需要使用的处理器
   - **Style Transfer**: 风格迁移处理
   - **Content Blend**: 内容融合处理
   - **Quality Enhancer**: 画质增强处理
   - **Portrait Editor**: 肖像编辑处理
   - **Frame Colorizer**: 视频上色
   - **Frame Enhancer**: 帧增强
   - **Lip Syncer**: 唇形同步
   - **Expression Restorer**: 表情恢复
   - **Age Modifier**: 年龄修改
   - **Background Remover**: 背景移除
4. **START**: 点击开始处理
5. **OUTPUT**: 下载处理后的结果文件

### 推荐设置
- **视频长度**: 免费Colab建议视频长度不超过10-15秒
- **输出格式**: 音频编码建议选择AAC以获得更好的兼容性
- **分辨率**: 根据GPU显存大小选择合适的分辨率
- **检测角度**: 如果目标视频包含侧面镜头，勾选90度检测角度

### Preview失败排查
1. 检查日志中是否有错误信息
2. 确保已上传SOURCE和TARGET文件
3. 尝试点击REFESH预览按钮
4. 如果问题持续，可以尝试重启运行时

## ⚠️ 注意事项

- 免费Colab会话有使用时间限制（通常12小时），长时间运行可能会被中断
- GPU资源由系统动态分配，可能获得T4、P100或V100等不同型号
- 处理大型视频文件时请确保有足够的存储空间（Colab提供约100GB）
- 建议在处理完成后及时下载结果文件到本地
- 首次运行会自动下载模型文件，可能需要几分钟时间
- Cloudflare Tunnel生成的链接每次启动都会变化

## 📝 故障排除

### 常见问题
1. **GPU内存不足**: 尝试降低视频分辨率或缩短视频长度
2. **依赖安装失败**: 尝试重新运行安装代码单元
3. **会话超时**: 保持浏览器窗口打开，或重新连接运行时
4. **连接被拒绝**: 等待Cloudflare Tunnel生成链接（通常需要5-10秒）
5. **模型下载失败**: 检查网络连接，或尝试重新启动运行时

### 重新启动
如果遇到问题，可以点击菜单栏 **Runtime > Restart runtime** 重新启动，然后重新执行所有代码单元。

## 📊 性能参考

| GPU型号 | 显存 | 适用场景 |
|---------|------|----------|
| T4 | 16GB | 标准视频处理 |
| P100 | 16GB | 高清视频处理 |
| V100 | 16/32GB | 大型视频处理 |

*GPU型号由Colab系统自动分配，无法手动选择。*

## 🔗 快速打开

点击以下链接在Colab中打开此笔记本：
```
https://colab.research.google.com/github/tjwcold/video/blob/main/videokit_colab.ipynb
```